# AFML Part 1 — Denoising Corrupted Weights

**TEAM ID and Name:** 26, Biased Bozos  
**TEAM MEMBERS:** 
- Arvind Ajay Bharadwaj - PES1UG24AM052
- Atharva Gore - PES1UG24AM346
- Bhooshan Deshpande - PES1UG24AM351
- Deeptanshu Kumar - PES1UG24AM357

We compare a noisy baseline, linear denoisers, a small neural network, a scale-aware linear denoiser, and convolutional residual correctors.

The final cell writes the two Part 1 outputs:

- `denoised_test_weights.npy`
- `submission_part1.csv`


## 1. Load the data


In [1]:
import numpy as np
from pathlib import Path
from scipy.signal import fftconvolve
from sklearn.linear_model import Ridge

TEAM_FOLDER = "26_Biased_Bozos"

def team_file(filename):
    matches = [
        p for p in Path("/kaggle/input").rglob(filename)
        if p.parent.name == TEAM_FOLDER
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one {filename} inside {TEAM_FOLDER}, "
            f"found {matches}"
        )

    print(filename, "->", matches[0])
    return matches[0]


clean = np.load(
    team_file("train_weights_clean.npy")
).astype(np.float32, copy=False)

noisy = np.load(
    team_file("train_weights_noisy.npy")
).astype(np.float32, copy=False)

test_noisy = np.load(
    team_file("test_weights_noisy.npy")
).astype(np.float32, copy=False)

n = len(test_noisy)

x_first, x_second = noisy[:n], noisy[n:]
y_first, y_second = clean[:n], clean[n:]

print()
print("clean:", clean.shape)
print("noisy:", noisy.shape)
print("test :", test_noisy.shape)


train_weights_clean.npy -> /kaggle/input/competitions/afml-orange-problem-denoising-transformer-weights/26_Biased_Bozos/train_weights_clean.npy
train_weights_noisy.npy -> /kaggle/input/competitions/afml-orange-problem-denoising-transformer-weights/26_Biased_Bozos/train_weights_noisy.npy
test_weights_noisy.npy -> /kaggle/input/competitions/afml-orange-problem-denoising-transformer-weights/26_Biased_Bozos/test_weights_noisy.npy

clean: (5923260,)
noisy: (5923260,)
test : (2961630,)


## 2. Make an 80/20 development split


In [2]:
rng = np.random.default_rng(42)
block_size = 32_768
gap = 2_048

# Keep positions away from the edges of each block and vector.
eligible = np.arange(gap, n - gap, dtype=np.int32)
inside_block = eligible % block_size
eligible = eligible[(inside_block >= gap) & (inside_block < block_size - gap)]

block_number = eligible // block_size
train_pool = eligible[block_number % 5 != 0]
val_pool = eligible[block_number % 5 == 0]
train_pos = rng.choice(train_pool, size=160_000, replace=False)
val_pos = rng.choice(val_pool, size=40_000, replace=False)

print(f"Training positions: {len(train_pos):,}; validation positions: {len(val_pos):,}")
print(f"Distinct blocks: {len(np.unique(train_pos // block_size))} train, "
      f"{len(np.unique(val_pos // block_size))} validation")


Training positions: 160,000; validation positions: 40,000
Distinct blocks: 72 train, 19 validation


## 3. Check the corruption pattern

The clean weights are nearly uncorrelated at offsets 1 and 128, but the corruption is correlated there. That motivates stencils that use neighboring noisy values at those offsets.


In [3]:
error = x_first - y_first
for offset in (1, 128):
    clean_corr = np.corrcoef(y_first[:-offset], y_first[offset:])[0, 1]
    error_corr = np.corrcoef(error[:-offset], error[offset:])[0, 1]
    print(f"Offset {offset:>3}: clean correlation = {clean_corr:.3f}, "
          f"corruption correlation = {error_corr:.3f}")


Offset   1: clean correlation = 0.001, corruption correlation = 0.342
Offset 128: clean correlation = 0.001, corruption correlation = 0.287


## 4. Shared helpers and baseline

A `side × side` stencil uses a square neighborhood with row spacing 128. Ridge regression learns one weight for each neighbor. RMSE is measured only on the held-out blocks.


In [4]:
def rmse(prediction, target):
    return float(np.sqrt(np.mean((prediction - target) ** 2, dtype=np.float64)))


def stencil_offsets(side):
    radius = side // 2
    return np.array([row * 128 + col
                     for row in range(-radius, radius + 1)
                     for col in range(-radius, radius + 1)], dtype=np.int32)


def make_features(vector, positions, offsets):
    return np.column_stack([vector[positions + offset] for offset in offsets])


scores = {"No denoising": rmse(x_first[val_pos], y_first[val_pos])}
print(f"Noisy baseline validation RMSE: {scores['No denoising']:.5f}")


Noisy baseline validation RMSE: 0.62051


## 5. Compare ordinary ridge denoisers

We try a pointwise model and two neighborhood sizes. Ridge regularization is fixed at `alpha=1`; only the neighborhood size changes.


In [5]:
ridge_models = {}
for side in (1, 13, 21):
    offsets = stencil_offsets(side)
    X_train = make_features(x_first, train_pos, offsets)
    X_val = make_features(x_first, val_pos, offsets)
    model = Ridge(alpha=1.0).fit(X_train, y_first[train_pos])
    name = f"{side}x{side} ridge"
    ridge_models[side] = model
    scores[name] = rmse(model.predict(X_val), y_first[val_pos])
    print(f"{name:<20} validation RMSE: {scores[name]:.5f}")
    del X_train, X_val


1x1 ridge            validation RMSE: 0.48205
13x13 ridge          validation RMSE: 0.07821
21x21 ridge          validation RMSE: 0.07495


## 6. Try a small neural network

This network starts from the 13×13 ridge prediction and learns a nonlinear correction. Its validation RMSE is checked after each epoch; training stops after three epochs without improvement. This keeps the comparison small and limits overfitting.


In [6]:
import torch
from torch import nn

torch.manual_seed(42)
torch.set_num_threads(min(4, torch.get_num_threads()))

class SmallMLP(nn.Module):
    def __init__(self, ridge_model):
        super().__init__()
        self.linear = nn.Linear(169, 1)
        self.correction = nn.Sequential(nn.Linear(169, 64), nn.ReLU(), nn.Linear(64, 1))
        # Start at the ridge solution. The network initially adds no correction.
        self.linear.weight.data.copy_(torch.from_numpy(ridge_model.coef_[None, :]))
        self.linear.bias.data.fill_(float(ridge_model.intercept_))
        nn.init.zeros_(self.correction[-1].weight)
        nn.init.zeros_(self.correction[-1].bias)

    def forward(self, values):
        return self.linear(values) + self.correction(values)

mlp_offsets = stencil_offsets(13)
X_train = torch.from_numpy(make_features(x_first, train_pos, mlp_offsets))
X_val = torch.from_numpy(make_features(x_first, val_pos, mlp_offsets))
t_train = torch.from_numpy(y_first[train_pos, None].copy())
t_val = torch.from_numpy(y_first[val_pos, None].copy())
mlp = SmallMLP(ridge_models[13])
optimizer = torch.optim.AdamW(mlp.parameters(), lr=0.001, weight_decay=0.0001)


In [7]:
best_rmse = float("inf")
wait = 0
for epoch in range(1, 16):
    mlp.train()
    for batch in torch.randperm(len(X_train)).split(4096):
        loss = ((mlp(X_train[batch]) - t_train[batch]) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    mlp.eval()
    with torch.no_grad():
        val_rmse = torch.sqrt(((mlp(X_val) - t_val) ** 2).mean()).item()
    if val_rmse < best_rmse - 0.00001:
        best_rmse = val_rmse
        best_state = {key: value.detach().clone() for key, value in mlp.state_dict().items()}
        wait = 0
    else:
        wait += 1
    if wait == 3:
        break

mlp.load_state_dict(best_state)
with torch.no_grad():
    train_rmse = torch.sqrt(((mlp(X_train) - t_train) ** 2).mean()).item()
scores["Small MLP"] = best_rmse
print(f"Small MLP: train RMSE = {train_rmse:.5f}, "
      f"validation RMSE = {best_rmse:.5f}, stopped at epoch {epoch}")
del X_train, X_val, t_train, t_val


Small MLP: train RMSE = 0.07779, validation RMSE = 0.07900, stopped at epoch 6


## 7. Try a scale-aware ridge denoiser

The noisy vector has two clear scales: some blocks have standard deviation near 1.3 and others near 0.15. A single filter must compromise between them. We use the noisy block standard deviation (available at test time) to choose between two 21×21 ridge models. The threshold `0.5` lies in the wide gap between these scales.


In [8]:
def block_stds(vector):
    return np.array([np.std(vector[start:start + block_size])
                     for start in range(0, len(vector), block_size)])


def large_scale_positions(positions, block_scales):
    return block_scales[positions // block_size] > 0.5


first_scales = block_stds(x_first)
train_large = large_scale_positions(train_pos, first_scales)
val_large = large_scale_positions(val_pos, first_scales)
offsets_21 = stencil_offsets(21)
scale_models = {}
scale_val_prediction = np.empty(len(val_pos), dtype=np.float32)
scale_train_prediction = np.empty(len(train_pos), dtype=np.float32)

for large in (False, True):
    # Fit each model only on the matching training blocks.
    tr = train_large == large
    va = val_large == large
    X_train = make_features(x_first, train_pos[tr], offsets_21)
    X_val = make_features(x_first, val_pos[va], offsets_21)
    model = Ridge(alpha=1.0).fit(X_train, y_first[train_pos[tr]])
    scale_models[large] = model
    scale_train_prediction[tr] = model.predict(X_train)
    scale_val_prediction[va] = model.predict(X_val)
    del X_train, X_val

scores["Scale-aware 21x21 ridge"] = rmse(scale_val_prediction, y_first[val_pos])
print(f"Scale-aware ridge: train RMSE = {rmse(scale_train_prediction, y_first[train_pos]):.5f}, "
      f"validation RMSE = {scores['Scale-aware 21x21 ridge']:.5f}")


Scale-aware ridge: train RMSE = 0.06769, validation RMSE = 0.06900


## 8. Select by validation RMSE

The second vector has not influenced this choice. Similar training and validation RMSE also helps rule out a large fitting gap.


In [9]:
print(f"{'Model':<27} {'Validation RMSE':>15}")
for name, score in sorted(scores.items(), key=lambda item: item[1]):
    print(f"{name:<27} {score:>15.5f}")

winner = min(scores, key=scores.get)
print(f"Selected: {winner}")


Model                       Validation RMSE
Scale-aware 21x21 ridge             0.06900
21x21 ridge                         0.07495
13x13 ridge                         0.07821
Small MLP                           0.07900
1x1 ridge                           0.48205
No denoising                        0.62051
Selected: Scale-aware 21x21 ridge


## 9. Check transfer to the second complete vector

For full-vector prediction, a sparse convolution applies the ridge stencil efficiently. This evaluation includes all weights, including block boundaries. The second vector is used only here, after model selection.


In [10]:
def predict_full_vector(vector, model, offsets):
    radius = int(np.max(np.abs(offsets)))
    kernel = np.zeros(2 * radius + 1, dtype=np.float32)
    kernel[radius - offsets] = model.coef_.astype(np.float32)
    return (fftconvolve(vector, kernel, mode="same") + model.intercept_).astype(np.float32)


def predict_with_scale_models(vector, models):
    scales = block_stds(vector)
    large = scales[np.arange(len(vector)) // block_size] > 0.5
    prediction = np.empty(len(vector), dtype=np.float32)
    for group in (False, True):
        all_values = predict_full_vector(vector, models[group], offsets_21)
        prediction[large == group] = all_values[large == group]
    return prediction


assert winner == "Scale-aware 21x21 ridge", "Update the final prediction cells if another model wins."
second_prediction = predict_with_scale_models(x_second, scale_models)
print(f"Second-vector noisy RMSE: {rmse(x_second, y_second):.5f}")
print(f"Second-vector denoised RMSE: {rmse(second_prediction, y_second):.5f}")


Second-vector noisy RMSE: 0.60749
Second-vector denoised RMSE: 0.06836


## 10. Refit the selected ridge model on both pairs

Model choice is finished, so the development split and second vector can both contribute to the final fit. We sample equally from the two paired vectors. The resulting test prediction stays in memory; the final submission files are written only in the last cell.


In [11]:
fit_first = rng.choice(eligible, size=120_000, replace=False)
fit_second = n + rng.choice(eligible, size=120_000, replace=False)
fit_positions = np.concatenate([fit_first, fit_second])

# The two paired vectors and test vector share the same weight layout.
second_scales = block_stds(x_second)
fit_scales = np.stack([first_scales, second_scales])
fit_large = fit_scales[fit_positions // n, (fit_positions % n) // block_size] > 0.5

final_models = {}
for large in (False, True):
    selected = fit_positions[fit_large == large]
    X = make_features(noisy, selected, offsets_21)
    final_models[large] = Ridge(alpha=1.0).fit(X, clean[selected])
    del X

# This is the current best test prediction. Later cells may improve it.
denoised_test = predict_with_scale_models(test_noisy, final_models)
final_method_name = "Scale-aware 21x21 Ridge"

assert denoised_test.shape == test_noisy.shape
assert denoised_test.dtype == np.float32
assert np.isfinite(denoised_test).all()

print(f"Created ridge test prediction: {denoised_test.shape}, {denoised_test.dtype}")


Created ridge test prediction: (2961630,), float32


## 11. Published convolutional denoisers

The MLP predicts one weight at a time. Convolutional models share filters across nearby weights. We test two **small adaptations** of published denoisers:

- [DnCNN](https://arxiv.org/abs/1608.03981) ([authors’ project](https://github.com/cszn/DnCNN)): stacked convolutions learn a residual correction.
- [RED-Net](https://arxiv.org/abs/1603.09056): an encoder–decoder with skip connections restores a corrupted input.

We arrange weights in rows of 128 because the corruption is correlated at offset 128. Each network learns a correction to the scale-aware ridge prediction. We train only on high-scale blocks, where that prediction has the most error; low-scale blocks keep the ridge output. This is an adaptation to weight data, not a reproduction of the image-denoising results. Published [diffusion-based weight generation](https://arxiv.org/abs/2407.11966) uses collections of trained networks. With only two paired vectors and an RMSE objective, we test the supervised denoisers here.


In [12]:
# Make 64-row patches from complete blocks. Validation blocks stay separate.
def as_image(vector):
    return np.pad(vector, (0, (-len(vector)) % 128)).reshape(-1, 128)


def high_scale_patches(x, ridge, y, split_for_validation):
    x_image, ridge_image, y_image = map(as_image, (x, ridge, y))
    large = block_stds(x) > 0.5
    training, validation = [], []
    for block in range(len(x) // block_size):
        if not large[block]:
            continue
        for part in range(4):
            rows = slice(block * 256 + part * 64, block * 256 + (part + 1) * 64)
            inputs = np.stack([x_image[rows], ridge_image[rows]])
            target = (y_image[rows] - ridge_image[rows])[None]
            destination = validation if split_for_validation and block % 5 == 0 else training
            destination.append((inputs, target))
    return training, validation


def patch_tensors(patches):
    x = torch.from_numpy(np.stack([item[0] for item in patches]).astype(np.float32))
    y = torch.from_numpy(np.stack([item[1] for item in patches]).astype(np.float32))
    return x, y

first_ridge = predict_with_scale_models(x_first, scale_models)
train_patches, val_patches = high_scale_patches(x_first, first_ridge, y_first, True)
cnn_X_train, cnn_y_train = patch_tensors(train_patches)
cnn_X_val, cnn_y_val = patch_tensors(val_patches)
patch_ridge_rmse = torch.sqrt(cnn_y_val[:, :, 8:-8, 8:-8].square().mean()).item()
print(f"High-scale patches: {len(train_patches)} train, {len(val_patches)} validation")
print(f"Ridge validation patch RMSE: {patch_ridge_rmse:.5f}")


High-scale patches: 148 train, 40 validation
Ridge validation patch RMSE: 0.08920


### Model definitions

Both models start at zero correction, so their initial prediction equals the ridge baseline. DnCNN uses dilated convolutions for a wider view. The smaller RED-Net compresses and reconstructs each patch with skip connections.


In [13]:
class DnCNNCorrection(nn.Module):
    def __init__(self):
        super().__init__()
        layers = [nn.Conv2d(2, 16, 3, padding=1), nn.ReLU()]
        for dilation in (2, 3, 2, 1):
            layers += [nn.Conv2d(16, 16, 3, padding=dilation, dilation=dilation), nn.ReLU()]
        layers += [nn.Conv2d(16, 1, 3, padding=1)]
        self.layers = nn.Sequential(*layers)
        nn.init.zeros_(self.layers[-1].weight)
        nn.init.zeros_(self.layers[-1].bias)

    def forward(self, x):
        return self.layers(x)


class REDNetCorrection(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(2, 16, 3, padding=1), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(16, 24, 3, stride=2, padding=1), nn.ReLU())
        self.enc3 = nn.Sequential(nn.Conv2d(24, 32, 3, stride=2, padding=1), nn.ReLU())
        self.dec2 = nn.Sequential(nn.ConvTranspose2d(32, 24, 4, stride=2, padding=1), nn.ReLU())
        self.dec1 = nn.Sequential(nn.ConvTranspose2d(24, 16, 4, stride=2, padding=1), nn.ReLU())
        self.output = nn.Conv2d(16, 1, 3, padding=1)
        nn.init.zeros_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, x):
        skip1 = self.enc1(x)
        skip2 = self.enc2(skip1)
        middle = self.enc3(skip2)
        return self.output(self.dec1(self.dec2(middle) + skip2) + skip1)


### Train and compare held-out patches

The eight-pixel patch border is excluded from the loss because it lacks context. Early stopping selects the lowest validation RMSE and stops after four epochs without improvement.


In [14]:
# ---------------------------------------------------------
# GPU setup
# ---------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

# Move the dataset to GPU once.
cnn_X_train = cnn_X_train.to(device)
cnn_y_train = cnn_y_train.to(device)

cnn_X_val = cnn_X_val.to(device)
cnn_y_val = cnn_y_val.to(device)


def fit_cnn(model_type, seed, max_epochs=30):

    # Reproducibility
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Create model directly on GPU.
    model = model_type().to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    best_rmse = float("inf")
    stale = 0
    best_epoch = 0
    best_weights = None

    for epoch in range(1, max_epochs + 1):

        # -------------------------------------------------
        # Training
        # -------------------------------------------------

        model.train()

        # Generate permutation directly on the GPU.
        permutation = torch.randperm(
            len(cnn_X_train),
            device=device
        )

        for batch in permutation.split(4):

            prediction = model(
                cnn_X_train[batch]
            )

            error = (
                prediction
                - cnn_y_train[batch]
            )

            # Ignore an 8-pixel border.
            inner = error[
                :, :, 8:-8, 8:-8
            ]

            loss = inner.square().mean()

            optimizer.zero_grad(
                set_to_none=True
            )

            loss.backward()

            optimizer.step()

        # -------------------------------------------------
        # Validation
        # -------------------------------------------------

        model.eval()

        squared_error = 0.0
        count = 0

        with torch.no_grad():

            indices = torch.arange(
                len(cnn_X_val),
                device=device
            )

            for batch in indices.split(4):

                prediction = model(
                    cnn_X_val[batch]
                )

                error = (
                    prediction
                    - cnn_y_val[batch]
                )

                inner = error[
                    :, :, 8:-8, 8:-8
                ]

                squared_error += (
                    inner.square()
                    .sum()
                    .item()
                )

                count += inner.numel()

        validation_rmse = (
            squared_error / count
        ) ** 0.5

        print(
            f"Epoch {epoch:02d} | "
            f"Val RMSE: {validation_rmse:.5f}"
        )

        # -------------------------------------------------
        # Early stopping
        # -------------------------------------------------

        if validation_rmse < best_rmse - 0.00001:

            best_rmse = validation_rmse
            best_epoch = epoch
            stale = 0

            # Store best weights on CPU so we don't waste
            # extra GPU memory.
            best_weights = {
                key: value.detach().cpu().clone()
                for key, value
                in model.state_dict().items()
            }

        else:
            stale += 1

            if stale == 4:
                print("Early stopping.")
                break

    # Restore best checkpoint.
    model.load_state_dict(
        best_weights
    )

    model.eval()

    return model, best_rmse, best_epoch


# ---------------------------------------------------------
# Train the candidate architectures
# ---------------------------------------------------------

deep_models = {}

for name, model_type, seed in (
    ("DnCNN correction", DnCNNCorrection, 123),
    ("RED-Net correction", REDNetCorrection, 456),
):

    print("\n", "=" * 50)
    print(name)
    print("=" * 50)

    model, patch_rmse, epoch = fit_cnn(
        model_type,
        seed
    )

    deep_models[name] = (
        model,
        epoch
    )

    print(
        f"{name}: "
        f"patch RMSE {patch_rmse:.5f}, "
        f"best epoch {epoch}"
    )

Using device: cuda

DnCNN correction
Epoch 01 | Val RMSE: 0.08916
Epoch 02 | Val RMSE: 0.08297
Epoch 03 | Val RMSE: 0.07156
Epoch 04 | Val RMSE: 0.06785
Epoch 05 | Val RMSE: 0.06546
Epoch 06 | Val RMSE: 0.06380
Epoch 07 | Val RMSE: 0.06270
Epoch 08 | Val RMSE: 0.06185
Epoch 09 | Val RMSE: 0.06128
Epoch 10 | Val RMSE: 0.06090
Epoch 11 | Val RMSE: 0.06056
Epoch 12 | Val RMSE: 0.06025
Epoch 13 | Val RMSE: 0.06002
Epoch 14 | Val RMSE: 0.05980
Epoch 15 | Val RMSE: 0.05966
Epoch 16 | Val RMSE: 0.05953
Epoch 17 | Val RMSE: 0.05941
Epoch 18 | Val RMSE: 0.05931
Epoch 19 | Val RMSE: 0.05918
Epoch 20 | Val RMSE: 0.05908
Epoch 21 | Val RMSE: 0.05895
Epoch 22 | Val RMSE: 0.05896
Epoch 23 | Val RMSE: 0.05892
Epoch 24 | Val RMSE: 0.05897
Epoch 25 | Val RMSE: 0.05889
Epoch 26 | Val RMSE: 0.05882
Epoch 27 | Val RMSE: 0.05888
Epoch 28 | Val RMSE: 0.05875
Epoch 29 | Val RMSE: 0.05868
Epoch 30 | Val RMSE: 0.05874
DnCNN correction: patch RMSE 0.05868, best epoch 29

RED-Net correction
Epoch 01 | Val RMSE: 

### Compare on the same 40,000 validation positions

Overlapping row tiles remove seams when predicting a full vector. Corrections are applied only in high-scale blocks.


In [15]:
def cnn_correction(vector, ridge_vector, model):
    """
    Apply the CNN correction model to the full flattened weight vector.

    The model may live on CPU or GPU; this function automatically uses
    the same device as the model.
    """

    # Infer the device directly from the model.
    device = next(model.parameters()).device

    x_image, ridge_image = map(
        as_image,
        (vector, ridge_vector)
    )

    result = np.empty_like(
        x_image,
        dtype=np.float32
    )

    model.eval()

    with torch.no_grad():

        for top in range(0, len(x_image), 256):

            bottom = min(
                top + 256,
                len(x_image)
            )

            # Add context above/below the current tile.
            start = max(0, top - 16)
            stop = min(
                len(x_image),
                bottom + 16
            )

            # Two input channels:
            #   channel 0 = noisy weights
            #   channel 1 = ridge prediction
            tile = np.stack([
                x_image[start:stop],
                ridge_image[start:stop]
            ])[None]

            # RED-Net downsamples twice, so the image height
            # must be divisible by 4.
            padding = (
                -tile.shape[-2]
            ) % 4

            if padding > 0:
                tile = np.pad(
                    tile,
                    (
                        (0, 0),     # batch
                        (0, 0),     # channels
                        (0, padding),  # height
                        (0, 0),     # width
                    )
                )

            # NumPy -> PyTorch -> GPU
            tile_tensor = torch.from_numpy(
                tile.astype(
                    np.float32,
                    copy=False
                )
            ).to(
                device,
                non_blocking=True
            )

            # Forward pass on GPU
            output = model(
                tile_tensor
            )[0, 0]

            # GPU -> CPU -> NumPy
            output = (
                output
                .detach()
                .cpu()
                .numpy()
            )

            # Only keep the center region corresponding
            # to this tile, excluding the added context.
            result[top:bottom] = output[
                top - start:
                bottom - start
            ]

    return result.reshape(-1)[:len(vector)]


def apply_cnn(vector, ridge_vector, model):
    """
    Apply CNN correction only to high-scale blocks.
    Low-scale blocks retain the Ridge prediction.
    """

    prediction = ridge_vector.copy()

    large_blocks = (
        block_stds(vector) > 0.5
    )

    large_positions = large_blocks[
        np.arange(len(vector))
        // block_size
    ]

    correction = cnn_correction(
        vector,
        ridge_vector,
        model
    )

    prediction[
        large_positions
    ] += correction[
        large_positions
    ]

    return prediction


# ---------------------------------------------------------
# Evaluate trained CNN models
# ---------------------------------------------------------

deep_scores = {}

for name, (model, _) in deep_models.items():

    print(
        f"\nEvaluating {name} "
        f"on {next(model.parameters()).device}"
    )

    prediction = apply_cnn(
        x_first,
        first_ridge,
        model
    )

    deep_scores[name] = rmse(
        prediction[val_pos],
        y_first[val_pos]
    )

    print(
        f"{name}: "
        f"full validation RMSE "
        f"{deep_scores[name]:.5f}"
    )


best_deep_name = min(
    deep_scores,
    key=deep_scores.get
)

print(
    f"\nRidge baseline on same positions: "
    f"{rmse(first_ridge[val_pos], y_first[val_pos]):.5f}"
)

print(
    f"Selected convolutional model: "
    f"{best_deep_name}"
)


Evaluating DnCNN correction on cuda:0
DnCNN correction: full validation RMSE 0.04976

Evaluating RED-Net correction on cuda:0
RED-Net correction: full validation RMSE 0.05086

Ridge baseline on same positions: 0.06900
Selected convolutional model: DnCNN correction


### Check transfer to the untouched second vector

The convolutional model is selected using the 80/20 validation blocks. We check its RMSE on the second complete vector only after selection.


In [16]:
best_deep_model, best_deep_epoch = deep_models[best_deep_name]
second_deep = apply_cnn(x_second, second_prediction, best_deep_model)
ridge_transfer_rmse = rmse(second_prediction, y_second)
deep_transfer_rmse = rmse(second_deep, y_second)
print(f"Second-vector ridge RMSE: {ridge_transfer_rmse:.5f}")
print(f"Second-vector {best_deep_name} RMSE: {deep_transfer_rmse:.5f}")
use_deep_model = (deep_scores[best_deep_name] < rmse(first_ridge[val_pos], y_first[val_pos])
                  and deep_transfer_rmse < ridge_transfer_rmse)
print(f"Use convolutional model for test weights: {use_deep_model}")


Second-vector ridge RMSE: 0.06836
Second-vector DnCNN correction RMSE: 0.04931
Use convolutional model for test weights: True


## 12. Refit the selected convolutional model

When the convolutional model improves both checks, retrain it on the high-scale blocks of both paired vectors for the selected number of epochs. The ridge prediction remains the first stage. The final submission files are still written only in the last cell.


In [17]:
if use_deep_model:
    final_patches = []

    for x_part, y_part in (
        (x_first, y_first),
        (x_second, y_second),
    ):
        ridge_part = predict_with_scale_models(
            x_part,
            final_models
        )

        patches, _ = high_scale_patches(
            x_part,
            ridge_part,
            y_part,
            False
        )

        final_patches.extend(patches)

    # Convert patches to tensors.
    final_X, final_y = patch_tensors(
        final_patches
    )

    # Move training tensors to GPU.
    final_X = final_X.to(device)
    final_y = final_y.to(device)

    model_type = (
        DnCNNCorrection
        if best_deep_name.startswith("DnCNN")
        else REDNetCorrection
    )

    seed = (
        123
        if model_type is DnCNNCorrection
        else 456
    )

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Create model on GPU.
    final_cnn = model_type().to(device)

    optimizer = torch.optim.Adam(
        final_cnn.parameters(),
        lr=0.001
    )

    for epoch in range(best_deep_epoch):

        final_cnn.train()

        # Generate permutation directly on GPU.
        permutation = torch.randperm(
            len(final_X),
            device=device
        )

        for batch in permutation.split(4):

            error = (
                final_cnn(final_X[batch])
                - final_y[batch]
            )

            loss = (
                error[:, :, 8:-8, 8:-8]
                .square()
                .mean()
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            loss.backward()
            optimizer.step()

        print(
            f"Final CNN epoch "
            f"{epoch + 1}/{best_deep_epoch}"
        )

    final_cnn.eval()

    # Ridge remains CPU/scikit-learn, which is fine.
    ridge_test = predict_with_scale_models(
        test_noisy,
        final_models
    )

    # apply_cnn() should be the GPU-aware version we just changed.
    denoised_test = apply_cnn(
        test_noisy,
        ridge_test,
        final_cnn
    ).astype(np.float32)

    final_method_name = (
        f"Scale-aware 21x21 Ridge + "
        f"{best_deep_name}"
    )

    assert (
        denoised_test.shape
        == test_noisy.shape
    )

    assert np.isfinite(
        denoised_test
    ).all()

    print(
        f"Created test prediction using "
        f"{best_deep_name} after ridge"
    )

    print(
        f"Final training patches: "
        f"{len(final_patches)}, "
        f"epochs: {best_deep_epoch}"
    )

else:
    print(
        "Convolutional correction did not "
        "beat ridge; retaining ridge prediction."
    )

Final CNN epoch 1/29
Final CNN epoch 2/29
Final CNN epoch 3/29
Final CNN epoch 4/29
Final CNN epoch 5/29
Final CNN epoch 6/29
Final CNN epoch 7/29
Final CNN epoch 8/29
Final CNN epoch 9/29
Final CNN epoch 10/29
Final CNN epoch 11/29
Final CNN epoch 12/29
Final CNN epoch 13/29
Final CNN epoch 14/29
Final CNN epoch 15/29
Final CNN epoch 16/29
Final CNN epoch 17/29
Final CNN epoch 18/29
Final CNN epoch 19/29
Final CNN epoch 20/29
Final CNN epoch 21/29
Final CNN epoch 22/29
Final CNN epoch 23/29
Final CNN epoch 24/29
Final CNN epoch 25/29
Final CNN epoch 26/29
Final CNN epoch 27/29
Final CNN epoch 28/29
Final CNN epoch 29/29
Created test prediction using DnCNN correction after ridge
Final training patches: 376, epochs: 29


## 13. Improve DnCNN with a two-model ensemble

[Deep ensembles](https://arxiv.org/abs/1612.01474) average predictions from independently trained networks. We test one additional DnCNN with a different random seed and average its output with the existing DnCNN. This changes no earlier experiment. The improvement must hold on both the original validation positions and the second weight vector before we use it for the test weights.

We also checked a wider convolutional receptive field and a second correction pass. The wider network was worse on both checks; the second pass improved RMSE by less than 0.0001. The two-model average gave the clearest gain, so only that method is added to the final pipeline.


In [18]:
# Train a second DnCNN on the same development blocks with an independent seed.
second_dncnn, second_patch_rmse, second_epoch = fit_cnn(DnCNNCorrection, seed=789)
print(f"Second DnCNN: patch RMSE {second_patch_rmse:.5f}, best epoch {second_epoch}")

first_dev_prediction = apply_cnn(x_first, first_ridge, deep_models["DnCNN correction"][0])
second_dev_prediction = apply_cnn(x_first, first_ridge, second_dncnn)
ensemble_dev_prediction = (first_dev_prediction + second_dev_prediction) / 2

second_transfer_prediction = apply_cnn(x_second, second_prediction, second_dncnn)
ensemble_transfer_prediction = (second_deep + second_transfer_prediction) / 2

single_val_rmse = rmse(first_dev_prediction[val_pos], y_first[val_pos])
ensemble_val_rmse = rmse(ensemble_dev_prediction[val_pos], y_first[val_pos])
single_transfer_rmse = rmse(second_deep, y_second)
ensemble_transfer_rmse = rmse(ensemble_transfer_prediction, y_second)

print(f"{'Model':<23} {'Validation RMSE':>15} {'Second-vector RMSE':>20}")
print(f"{'One DnCNN':<23} {single_val_rmse:>15.5f} {single_transfer_rmse:>20.5f}")
print(f"{'Two-model average':<23} {ensemble_val_rmse:>15.5f} {ensemble_transfer_rmse:>20.5f}")
use_ensemble = ensemble_val_rmse < single_val_rmse and ensemble_transfer_rmse < single_transfer_rmse
print(f"Use ensemble for test weights: {use_ensemble}")


Epoch 01 | Val RMSE: 0.08479
Epoch 02 | Val RMSE: 0.07480
Epoch 03 | Val RMSE: 0.06939
Epoch 04 | Val RMSE: 0.06566
Epoch 05 | Val RMSE: 0.06351
Epoch 06 | Val RMSE: 0.06264
Epoch 07 | Val RMSE: 0.06170
Epoch 08 | Val RMSE: 0.06137
Epoch 09 | Val RMSE: 0.06104
Epoch 10 | Val RMSE: 0.06084
Epoch 11 | Val RMSE: 0.06074
Epoch 12 | Val RMSE: 0.06054
Epoch 13 | Val RMSE: 0.06041
Epoch 14 | Val RMSE: 0.06024
Epoch 15 | Val RMSE: 0.06044
Epoch 16 | Val RMSE: 0.06036
Epoch 17 | Val RMSE: 0.06014
Epoch 18 | Val RMSE: 0.06009
Epoch 19 | Val RMSE: 0.06008
Epoch 20 | Val RMSE: 0.06008
Epoch 21 | Val RMSE: 0.06011
Epoch 22 | Val RMSE: 0.06008
Epoch 23 | Val RMSE: 0.06002
Epoch 24 | Val RMSE: 0.06018
Epoch 25 | Val RMSE: 0.06009
Epoch 26 | Val RMSE: 0.06008
Epoch 27 | Val RMSE: 0.06030
Early stopping.
Second DnCNN: patch RMSE 0.06002, best epoch 23
Model                   Validation RMSE   Second-vector RMSE
One DnCNN                       0.04976              0.04931
Two-model average              

### Refit the second DnCNN and create the ensemble prediction


In [19]:
if use_ensemble and use_deep_model:

    seed = 789

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Create the second DnCNN directly on the GPU.
    final_dncnn_2 = DnCNNCorrection().to(device)

    optimizer = torch.optim.Adam(
        final_dncnn_2.parameters(),
        lr=0.001
    )

    for epoch in range(second_epoch):

        final_dncnn_2.train()

        # Generate shuffled indices directly on the GPU.
        permutation = torch.randperm(
            len(final_X),
            device=device
        )

        for batch in permutation.split(4):

            prediction = final_dncnn_2(
                final_X[batch]
            )

            error = (
                prediction
                - final_y[batch]
            )

            loss = (
                error[:, :, 8:-8, 8:-8]
                .square()
                .mean()
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            loss.backward()
            optimizer.step()

        print(
            f"Second DnCNN epoch "
            f"{epoch + 1}/{second_epoch}"
        )

    final_dncnn_2.eval()

    # apply_cnn() should be the GPU-aware version,
    # which moves each inference tile to the model's device.
    second_test_prediction = apply_cnn(
        test_noisy,
        ridge_test,
        final_dncnn_2,
    )

    # Average the two independently trained DnCNN-corrected predictions.
    denoised_test = (
        (
            denoised_test
            + second_test_prediction
        )
        / 2
    ).astype(np.float32)

    final_method_name = (
        "Scale-aware 21x21 Ridge + "
        "two-DnCNN ensemble"
    )

    assert (
        denoised_test.shape
        == test_noisy.shape
    )

    assert np.isfinite(
        denoised_test
    ).all()

    print(
        f"Created ensemble test prediction: "
        f"{denoised_test.shape}, "
        f"{denoised_test.dtype}"
    )

else:
    print(
        "Retained the current "
        "non-ensemble test prediction."
    )

Second DnCNN epoch 1/23
Second DnCNN epoch 2/23
Second DnCNN epoch 3/23
Second DnCNN epoch 4/23
Second DnCNN epoch 5/23
Second DnCNN epoch 6/23
Second DnCNN epoch 7/23
Second DnCNN epoch 8/23
Second DnCNN epoch 9/23
Second DnCNN epoch 10/23
Second DnCNN epoch 11/23
Second DnCNN epoch 12/23
Second DnCNN epoch 13/23
Second DnCNN epoch 14/23
Second DnCNN epoch 15/23
Second DnCNN epoch 16/23
Second DnCNN epoch 17/23
Second DnCNN epoch 18/23
Second DnCNN epoch 19/23
Second DnCNN epoch 20/23
Second DnCNN epoch 21/23
Second DnCNN epoch 22/23
Second DnCNN epoch 23/23
Created ensemble test prediction: (2961630,), float32


## 14. Test the Bayesian compensation paper

[Shao, Liew, and Gündüz](https://arxiv.org/abs/2105.10699) model an observed weight as clean weight plus **independent Gaussian noise**. Their Gaussian MMSE estimate shrinks each observed weight toward the clean-weight mean. Their population and bias compensators change this into an affine mapping `a × observation + b`. The compensators are designed to improve a reconstructed network’s **inference accuracy**, which is a different goal from our weight RMSE. We do not have the network architecture or downstream examples needed to tune inference accuracy here.

Our corruption is correlated at offsets 1 and 128, so the paper’s independent-noise model is a simplification. We test (1) its Gaussian MMSE step on raw noisy weights and on the DnCNN ensemble, and (2) an affine mapping fitted for RMSE on the training blocks. The latter is a favorable RMSE-tuned test of the paper’s affine compensation family. We evaluate both on the same held-out blocks and on the second vector. Earlier cells and the saved ensemble weights remain unchanged.


In [20]:
def gaussian_mmse_coefficients(observed, clean_target, positions):
    # The paper's Gaussian-prior MMSE estimator (zero compensators).
    clean_sample = clean_target[positions]
    mean = float(clean_sample.mean())
    weight_variance = float(clean_sample.var())
    noise_variance = float(np.mean((observed[positions] - clean_sample) ** 2))
    slope = weight_variance / (weight_variance + noise_variance)
    intercept = (1 - slope) * mean
    return slope, intercept


def fitted_affine_coefficients(observed, clean_target, positions):
    # Best affine mapping on training blocks for our RMSE objective.
    x = observed[positions].astype(np.float64)
    y = clean_target[positions].astype(np.float64)
    slope = float(np.cov(x, y, bias=True)[0, 1] / np.var(x))
    intercept = float(y.mean() - slope * x.mean())
    return slope, intercept


for source_name, first_observation, second_observation in (
    ("Raw noisy", x_first, x_second),
    ("DnCNN ensemble", ensemble_dev_prediction, ensemble_transfer_prediction),
):
    print(f"\n{source_name}")
    methods = {
        "Unchanged": (1.0, 0.0),
        "Paper Gaussian MMSE": gaussian_mmse_coefficients(first_observation, y_first, train_pos),
        "RMSE-tuned affine": fitted_affine_coefficients(first_observation, y_first, train_pos),
    }
    for name, (slope, intercept) in methods.items():
        val_score = rmse(slope * first_observation[val_pos] + intercept, y_first[val_pos])
        transfer_score = rmse(slope * second_observation + intercept, y_second)
        print(f"{name:<20} validation={val_score:.6f}, second vector={transfer_score:.6f}, "
              f"slope={slope:.6f}, intercept={intercept:.6f}")



Raw noisy
Unchanged            validation=0.620508, second vector=0.607486, slope=1.000000, intercept=0.000000
Paper Gaussian MMSE  validation=0.482049, second vector=0.470507, slope=0.586642, intercept=-0.000106
RMSE-tuned affine    validation=0.482051, second vector=0.470513, slope=0.588433, intercept=-0.002721

DnCNN ensemble
Unchanged            validation=0.049030, second vector=0.048530, slope=1.000000, intercept=0.000000
Paper Gaussian MMSE  validation=0.049083, second vector=0.048584, slope=0.995606, intercept=-0.000001
RMSE-tuned affine    validation=0.049013, second vector=0.048520, slope=0.998873, intercept=-0.000820


**Result:** The paper’s Gaussian MMSE step improves raw weights but remains far behind the spatial DnCNN ensemble. Applying it after the ensemble increases RMSE. RMSE-tuned affine compensation changes the independent second-vector RMSE by only a few millionths, so it is not a meaningful improvement. Therefore the two-DnCNN prediction remains the final Part 1 prediction. The next cell writes the required Kaggle outputs.


## 15. Create the final Part 1 submission files

This is the **only authoritative saving cell** in the notebook.

It writes:

1. `denoised_test_weights.npy` — the denoised weight vector needed for Part 2.
2. `submission_part1.csv` — the Part 1 Kaggle leaderboard submission.

The assertions below verify the required vector length, CSV shape, IDs, column names, and absence of NaN/Inf values before writing the files.


In [21]:
import pandas as pd
from pathlib import Path

# Kaggle notebooks should write generated artifacts to /kaggle/working.
# Falling back to the current directory keeps this cell usable locally.
output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

final_weights = np.asarray(denoised_test, dtype=np.float32).reshape(-1)

# ------------------------------------------------------------------
# Final weight-vector sanity checks
# ------------------------------------------------------------------
assert final_weights.shape == (2_961_630,), (
    f"Expected 2,961,630 weights, got {final_weights.shape}"
)
assert final_weights.shape == test_noisy.shape
assert np.isfinite(final_weights).all(), "Final weights contain NaN or Inf."

# ------------------------------------------------------------------
# Save the denoised weights for Part 2
# ------------------------------------------------------------------
weights_path = output_dir / "denoised_test_weights.npy"
np.save(weights_path, final_weights)

# ------------------------------------------------------------------
# Create the Part 1 Kaggle submission
# ------------------------------------------------------------------
submission = pd.DataFrame({
    "id": np.arange(len(final_weights), dtype=np.int64),
    "weight": final_weights,
})

assert submission.shape == (2_961_630, 2)
assert list(submission.columns) == ["id", "weight"]
assert submission["id"].iloc[0] == 0
assert submission["id"].iloc[-1] == 2_961_629
assert submission["id"].is_unique
assert submission.isna().sum().sum() == 0
assert np.isfinite(submission["weight"].to_numpy()).all()

submission_path = output_dir / "submission_part1.csv"
submission.to_csv(submission_path, index=False)

# Kaggle requires this exact filename for notebook submission
submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

# ------------------------------------------------------------------
# Final report
# ------------------------------------------------------------------
print("=" * 60)
print("PART 1 SUBMISSION READY")
print("=" * 60)
print("Final method:", final_method_name)
print("Weights shape:", final_weights.shape)
print("Weights dtype:", final_weights.dtype)
print("CSV shape:", submission.shape)
print("NaN count:", int(submission.isna().sum().sum()))
print("First ID:", int(submission["id"].iloc[0]))
print("Last ID:", int(submission["id"].iloc[-1]))
print()
print("Created:")
print(weights_path)
print(submission_path)
print()
print("First five submission rows:")
display(submission.head())
print("Last five submission rows:")
display(submission.tail())


PART 1 SUBMISSION READY
Final method: Scale-aware 21x21 Ridge + two-DnCNN ensemble
Weights shape: (2961630,)
Weights dtype: float32
CSV shape: (2961630, 2)
NaN count: 0
First ID: 0
Last ID: 2961629

Created:
/kaggle/working/denoised_test_weights.npy
/kaggle/working/submission_part1.csv

First five submission rows:


,id,weight
0,0,0.262397
1,1,0.143805
2,2,2.034955
3,3,0.559684
4,4,-0.893003


Last five submission rows:


,id,weight
2961625,2961625,0.754208
2961626,2961626,1.097839
2961627,2961627,0.877733
2961628,2961628,0.853439
2961629,2961629,0.774303
